In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

import torch.nn as nn
import torch.optim as optim


from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, brier_score_loss, log_loss
)

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
base_path = "/content/drive/MyDrive/TFG"

train_csv = base_path + "/df_train_clean.csv"
valid_csv = base_path + "/df_valid_clean.csv"

df_train = pd.read_csv(train_csv)
df_valid = pd.read_csv(valid_csv)

print("Train:", len(df_train))
print("Valid:", len(df_valid))

Train: 12399
Valid: 3091


In [ ]:
urgent_cols = [
    "Pneumothorax",
    "Edema",
    "Pleural Effusion",
    "Enlarged Cardiomediastinum"
]

df_train["urgent"] = (df_train[urgent_cols] == 1).any(axis=1).astype(int)
df_valid["urgent"] = (df_valid[urgent_cols] == 1).any(axis=1).astype(int)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
class CheXpertDataset(Dataset):
    def __init__(self, dataframe, base_path, transform=None):
        self.df = dataframe
        self.base_path = base_path
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        relative_path = row["Path"].replace("CheXpert-v1.0/train/", "")
        img_path = os.path.join(self.base_path, relative_path)

        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        label = torch.tensor(row["urgent"], dtype=torch.float32)

        return img, label

In [ ]:

# COMPROBAR / DESCOMPRIMIR IMÁGENES


train_images_path = "/content/train_clean_images"
valid_images_path = "/content/valid_clean_images"

if not os.path.exists(train_images_path):
    !unzip -oq "/content/drive/MyDrive/TFG/train_clean_images.zip" -d "/content/"
    print("Train descomprimido")
else:
    print("Train ya estaba descomprimido")

if not os.path.exists(valid_images_path):
    !unzip -oq "/content/drive/MyDrive/TFG/valid_clean_images.zip" -d "/content/"
    print("Valid/Test descomprimido")
else:
    print("Valid/Test ya estaba descomprimido")

print("Existe train:", os.path.exists(train_images_path))
print("Existe valid/test:", os.path.exists(valid_images_path))


# CONFIGURACIÓN DEL EXPERIMENTO

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

output_dir = "/content/drive/MyDrive/TFG/modelo_robusto_5folds"
os.makedirs(output_dir, exist_ok=True)

configs = [
    {"config_name": "A_lr_0.001_ep10", "learning_rate": 0.001, "epochs": 10},
    {"config_name": "B_lr_0.0005_ep10", "learning_rate": 0.0005, "epochs": 10}
]

n_folds = 5
batch_size = 32

print("Configuraciones:")
for c in configs:
    print(c)

Train descomprimido
Valid/Test descomprimido
Existe train: True
Existe valid/test: True
Dispositivo: cuda
Configuraciones:
{'config_name': 'A_lr_0.001_ep10', 'learning_rate': 0.001, 'epochs': 10}
{'config_name': 'B_lr_0.0005_ep10', 'learning_rate': 0.0005, 'epochs': 10}


In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import confusion_matrix

def create_model():
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    model.fc = nn.Linear(in_features=2048, out_features=1)
    return model.to(device)


def train_one_epoch(model, train_loader, criterion, optimizer):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    return epoch_loss


def evaluate_model(model, data_loader, threshold=0.5):
    model.eval()

    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            probs = torch.sigmoid(outputs)
            preds = (probs >= threshold).float()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_labels = np.array(all_labels).ravel()
    all_preds = np.array(all_preds).ravel()
    all_probs = np.array(all_probs).ravel()

    metrics = {
        "accuracy": accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds, zero_division=0),
        "recall": recall_score(all_labels, all_preds, zero_division=0),
        "f1": f1_score(all_labels, all_preds, zero_division=0),
        "auc": roc_auc_score(all_labels, all_probs),
        "brier": brier_score_loss(all_labels, all_probs),
        "logloss": log_loss(all_labels, all_probs)
    }

    cm = confusion_matrix(all_labels, all_preds)

    return metrics, cm, all_labels, all_preds, all_probs

In [ ]:
results_path = os.path.join(output_dir, "robust_5fold_gridsearch_results.csv")

# Si ya existe un CSV de resultados, lo cargamos para no repetir folds completados
if os.path.exists(results_path):
    previous_results = pd.read_csv(results_path)
    completed = set(zip(previous_results["config_name"], previous_results["fold"]))
    all_results = previous_results.to_dict("records")
    print("Resultados previos cargados:", len(all_results))
else:
    completed = set()
    all_results = []

groups = df_train["patient_id"].values
X = df_train.index.values

group_kfold = GroupKFold(n_splits=n_folds)

for config in configs:
    config_name = config["config_name"]
    lr = config["learning_rate"]
    epochs = config["epochs"]

    print("\n" + "="*70)
    print(f"CONFIGURACIÓN: {config_name} | lr={lr} | epochs={epochs}")
    print("="*70)

    for fold, (train_idx, val_idx) in enumerate(group_kfold.split(X, df_train["urgent"], groups), start=1):

        if (config_name, fold) in completed:
            print(f"Saltando {config_name} - Fold {fold}, ya completado.")
            continue

        print("\n" + "-"*50)
        print(f"{config_name} - Fold {fold}/{n_folds}")
        print("-"*50)

        train_fold_df = df_train.iloc[train_idx].reset_index(drop=True)
        val_fold_df = df_train.iloc[val_idx].reset_index(drop=True)

        print("Train fold:", len(train_fold_df))
        print("Val fold:", len(val_fold_df))
        print("Distribución train:")
        print(train_fold_df["urgent"].value_counts())
        print("Distribución val:")
        print(val_fold_df["urgent"].value_counts())

        train_dataset = CheXpertDataset(train_fold_df, train_images_path, transform)
        val_dataset = CheXpertDataset(val_fold_df, train_images_path, transform)

        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=2,
            pin_memory=True
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=2,
            pin_memory=True
        )

        model = create_model()
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        train_losses = []

        for epoch in range(epochs):
            epoch_loss = train_one_epoch(model, train_loader, criterion, optimizer)
            train_losses.append(epoch_loss)

            print(f"{config_name} | Fold {fold} | Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f}")

        metrics, cm, labels, preds, probs = evaluate_model(model, val_loader, threshold=0.5)

        print("Métricas fold:")
        print(metrics)
        print("Matriz de confusión:")
        print(cm)

        # Guardar modelo del fold
        model_file = os.path.join(output_dir, f"model_{config_name}_fold_{fold}.pth")
        torch.save(model.state_dict(), model_file)

        # Guardar losses del fold
        losses_df = pd.DataFrame({
            "epoch": list(range(1, len(train_losses) + 1)),
            "loss": train_losses
        })
        losses_file = os.path.join(output_dir, f"losses_{config_name}_fold_{fold}.csv")
        losses_df.to_csv(losses_file, index=False)

        # Guardar labels, preds, probs del fold
        np.save(os.path.join(output_dir, f"labels_{config_name}_fold_{fold}.npy"), labels)
        np.save(os.path.join(output_dir, f"preds_{config_name}_fold_{fold}.npy"), preds)
        np.save(os.path.join(output_dir, f"probs_{config_name}_fold_{fold}.npy"), probs)

        # Guardar métricas del fold
        result = {
                    "config_name": config_name,
                    "learning_rate": lr,
                    "epochs": epochs,
                    "fold": fold,
                    "final_train_loss": train_losses[-1],
                    "accuracy": metrics["accuracy"],
                    "precision": metrics["precision"],
                    "recall": metrics["recall"],
                    "f1": metrics["f1"],
                    "auc": metrics["auc"],
                    "brier": metrics["brier"],
                    "logloss": metrics["logloss"],
                    "tn": cm[0, 0],
                    "fp": cm[0, 1],
                    "fn": cm[1, 0],
                    "tp": cm[1, 1],
                    "model_file": model_file,
                    "losses_file": losses_file
                }

        all_results.append(result)

        results_df = pd.DataFrame(all_results)
        results_df.to_csv(results_path, index=False)

        print(f"Guardado completado: {config_name} - Fold {fold}")

print("\nEXPERIMENTO ROBUSTO 5-FOLDS COMPLETADO")
results_df = pd.read_csv(results_path)
results_df


CONFIGURACIÓN: A_lr_0.001_ep10 | lr=0.001 | epochs=10

--------------------------------------------------
A_lr_0.001_ep10 - Fold 1/5
--------------------------------------------------
Train fold: 9919
Val fold: 2480
Distribución train:
urgent
0    5977
1    3942
Name: count, dtype: int64
Distribución val:
urgent
0    1478
1    1002
Name: count, dtype: int64
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 227MB/s]


A_lr_0.001_ep10 | Fold 1 | Epoch [1/10] - Loss: 0.5080
A_lr_0.001_ep10 | Fold 1 | Epoch [2/10] - Loss: 0.4562
A_lr_0.001_ep10 | Fold 1 | Epoch [3/10] - Loss: 0.4291
A_lr_0.001_ep10 | Fold 1 | Epoch [4/10] - Loss: 0.4199
A_lr_0.001_ep10 | Fold 1 | Epoch [5/10] - Loss: 0.4000
A_lr_0.001_ep10 | Fold 1 | Epoch [6/10] - Loss: 0.3774
A_lr_0.001_ep10 | Fold 1 | Epoch [7/10] - Loss: 0.3449
A_lr_0.001_ep10 | Fold 1 | Epoch [8/10] - Loss: 0.3077
A_lr_0.001_ep10 | Fold 1 | Epoch [9/10] - Loss: 0.2538
A_lr_0.001_ep10 | Fold 1 | Epoch [10/10] - Loss: 0.2139
Métricas fold:
{'accuracy': 0.7701612903225806, 'precision': 0.7014925373134329, 'recall': 0.7504990019960079, 'f1': 0.7251687560270009, 'auc': np.float64(0.8359387449728418), 'brier': np.float64(0.1734476706965339), 'logloss': 0.6079625166041351}
Matriz de confusión:
[[1158  320]
 [ 250  752]]
Guardado completado: A_lr_0.001_ep10 - Fold 1

--------------------------------------------------
A_lr_0.001_ep10 - Fold 2/5
----------------------------

,config_name,learning_rate,epochs,fold,final_train_loss,accuracy,precision,recall,f1,auc,brier,logloss,tn,fp,fn,tp,model_file,losses_file
0,A_lr_0.001_ep10,0.0010,10,1,0.213937,0.770161,0.701493,0.750499,0.725169,0.835939,0.173448,0.607963,1158,320,250,752,/content/drive/MyDrive/TFG/modelo_robusto_5fol...,/content/drive/MyDrive/TFG/modelo_robusto_5fol...
1,A_lr_0.001_ep10,0.0010,10,2,0.207130,0.788710,0.786047,0.665354,0.720682,0.831640,0.169400,0.635026,1280,184,340,676,/content/drive/MyDrive/TFG/modelo_robusto_5fol...,/content/drive/MyDrive/TFG/modelo_robusto_5fol...
2,A_lr_0.001_ep10,0.0010,10,3,0.227031,0.695565,0.568750,0.859391,0.684496,0.846032,0.229008,0.853518,906,621,134,819,/content/drive/MyDrive/TFG/modelo_robusto_5fol...,/content/drive/MyDrive/TFG/modelo_robusto_5fol...
3,A_lr_0.001_ep10,0.0010,10,4,0.205656,0.778226,0.732841,0.700303,0.716202,0.829240,0.172057,0.626456,1236,253,297,694,/content/drive/MyDrive/TFG/modelo_robusto_5fol...,/content/drive/MyDrive/TFG/modelo_robusto_5fol...
4,A_lr_0.001_ep10,0.0010,10,5,0.213535,0.730133,0.618831,0.829939,0.709004,0.833699,0.213385,0.893271,995,502,167,815,/content/drive/MyDrive/TFG/modelo_robusto_5fol...,/content/drive/MyDrive/TFG/modelo_robusto_5fol...
5,B_lr_0.0005_ep10,0.0005,10,1,0.061630,0.768952,0.707648,0.729541,0.718428,0.830665,0.193783,0.927307,1176,302,271,731,/content/drive/MyDrive/TFG/modelo_robusto_5fol...,/content/drive/MyDrive/TFG/modelo_robusto_5fol...
6,B_lr_0.0005_ep10,0.0005,10,2,0.065276,0.790726,0.823990,0.622047,0.708918,0.835876,0.186138,1.198609,1329,135,384,632,/content/drive/MyDrive/TFG/modelo_robusto_5fol...,/content/drive/MyDrive/TFG/modelo_robusto_5fol...
7,B_lr_0.0005_ep10,0.0005,10,3,0.066995,0.811290,0.808917,0.666317,0.730725,0.856017,0.163185,0.959827,1377,150,318,635,/content/drive/MyDrive/TFG/modelo_robusto_5fol...,/content/drive/MyDrive/TFG/modelo_robusto_5fol...
8,B_lr_0.0005_ep10,0.0005,10,4,0.062172,0.766935,0.799710,0.556004,0.655952,0.818326,0.202737,1.084505,1351,138,440,551,/content/drive/MyDrive/TFG/modelo_robusto_5fol...,/content/drive/MyDrive/TFG/modelo_robusto_5fol...
9,B_lr_0.0005_ep10,0.0005,10,5,0.061697,0.790641,0.740895,0.725051,0.732887,0.842874,0.177698,0.908938,1248,249,270,712,/content/drive/MyDrive/TFG/modelo_robusto_5fol...,/content/drive/MyDrive/TFG/modelo_robusto_5fol...


In [ ]:
results_df = pd.read_csv(results_path)

summary_df = results_df.groupby("config_name").agg({
    "accuracy": ["mean", "std"],
    "precision": ["mean", "std"],
    "recall": ["mean", "std"],
    "f1": ["mean", "std"],
    "auc": ["mean", "std"],
    "brier": ["mean", "std"],
    "logloss": ["mean", "std"],
    "final_train_loss": ["mean", "std"]
}).round(4)

summary_df.columns = ["_".join(col) for col in summary_df.columns]
summary_df = summary_df.reset_index()

summary_path = os.path.join(output_dir, "robust_5fold_summary_mean_std.csv")
summary_df.to_csv(summary_path, index=False)

summary_df

,config_name,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,auc_mean,auc_std,brier_mean,brier_std,logloss_mean,logloss_std,final_train_loss_mean,final_train_loss_std
0,A_lr_0.001_ep10,0.7526,0.0388,0.6816,0.0875,0.7611,0.0827,0.7111,0.0160,0.8353,0.0065,0.1915,0.0277,0.7232,0.1381,0.2135,0.0084
1,B_lr_0.0005_ep10,0.7857,0.0183,0.7762,0.0496,0.6598,0.0731,0.7094,0.0314,0.8368,0.0140,0.1847,0.0152,1.0158,0.1230,0.0636,0.0024


In [ ]:
thresholds = [0.5, 0.4, 0.3]
threshold_results = []

for _, row in results_df.iterrows():
    config_name = row["config_name"]
    fold = int(row["fold"])

    labels_file = os.path.join(output_dir, f"labels_{config_name}_fold_{fold}.npy")
    probs_file = os.path.join(output_dir, f"probs_{config_name}_fold_{fold}.npy")

    labels = np.load(labels_file).ravel()
    probs = np.load(probs_file).ravel()

    for th in thresholds:
        preds = (probs >= th).astype(int)

        threshold_results.append({
            "config_name": config_name,
            "fold": fold,
            "threshold": th,
            "accuracy": accuracy_score(labels, preds),
            "precision": precision_score(labels, preds, zero_division=0),
            "recall": recall_score(labels, preds, zero_division=0),
            "f1": f1_score(labels, preds, zero_division=0),
            "auc": roc_auc_score(labels, probs),
            "brier": brier_score_loss(labels, probs),
            "logloss": log_loss(labels, probs)
        })

threshold_df = pd.DataFrame(threshold_results)

threshold_path = os.path.join(output_dir, "robust_5fold_threshold_results_by_fold.csv")
threshold_df.to_csv(threshold_path, index=False)

threshold_summary_df = threshold_df.groupby(["config_name", "threshold"]).agg({
    "accuracy": ["mean", "std"],
    "precision": ["mean", "std"],
    "recall": ["mean", "std"],
    "f1": ["mean", "std"],
    "auc": ["mean", "std"],
    "brier": ["mean", "std"],
    "logloss": ["mean", "std"]
}).round(4)

threshold_summary_df.columns = ["_".join(col) for col in threshold_summary_df.columns]
threshold_summary_df = threshold_summary_df.reset_index()

threshold_summary_path = os.path.join(output_dir, "robust_5fold_threshold_summary_mean_std.csv")
threshold_summary_df.to_csv(threshold_summary_path, index=False)

threshold_summary_df

,config_name,threshold,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,auc_mean,auc_std,brier_mean,brier_std,logloss_mean,logloss_std
0,A_lr_0.001_ep10,0.3,0.7122,0.0574,0.6163,0.0817,0.8154,0.0748,0.6951,0.0274,0.8353,0.0065,0.1915,0.0277,0.7232,0.1381
1,A_lr_0.001_ep10,0.4,0.7351,0.0485,0.6502,0.0861,0.7875,0.0779,0.7048,0.0227,0.8353,0.0065,0.1915,0.0277,0.7232,0.1381
2,A_lr_0.001_ep10,0.5,0.7526,0.0388,0.6816,0.0875,0.7611,0.0827,0.7111,0.0160,0.8353,0.0065,0.1915,0.0277,0.7232,0.1381
3,B_lr_0.0005_ep10,0.3,0.7776,0.0189,0.7402,0.0524,0.6950,0.0738,0.7128,0.0246,0.8368,0.0140,0.1847,0.0152,1.0158,0.1230
4,B_lr_0.0005_ep10,0.4,0.7829,0.0191,0.7607,0.0502,0.6756,0.0728,0.7117,0.0291,0.8368,0.0140,0.1847,0.0152,1.0158,0.1230
5,B_lr_0.0005_ep10,0.5,0.7857,0.0183,0.7762,0.0496,0.6598,0.0731,0.7094,0.0314,0.8368,0.0140,0.1847,0.0152,1.0158,0.1230


In [ ]:

# DISCRIMINACIÓN


discrimination_table = summary_df[[
    "config_name",
    "accuracy_mean", "accuracy_std",
    "precision_mean", "precision_std",
    "recall_mean", "recall_std",
    "f1_mean", "f1_std",
    "auc_mean", "auc_std"
]].copy()

# Formato media ± std
discrimination_table["Accuracy"] = (
    discrimination_table["accuracy_mean"].round(3).astype(str)
    + " ± " +
    discrimination_table["accuracy_std"].round(3).astype(str)
)

discrimination_table["Precision"] = (
    discrimination_table["precision_mean"].round(3).astype(str)
    + " ± " +
    discrimination_table["precision_std"].round(3).astype(str)
)

discrimination_table["Recall"] = (
    discrimination_table["recall_mean"].round(3).astype(str)
    + " ± " +
    discrimination_table["recall_std"].round(3).astype(str)
)

discrimination_table["F1-score"] = (
    discrimination_table["f1_mean"].round(3).astype(str)
    + " ± " +
    discrimination_table["f1_std"].round(3).astype(str)
)

discrimination_table["AUC"] = (
    discrimination_table["auc_mean"].round(3).astype(str)
    + " ± " +
    discrimination_table["auc_std"].round(3).astype(str)
)

# Seleccionar columnas finales
discrimination_table = discrimination_table[[
    "config_name",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "AUC"
]]

discrimination_table

,config_name,Accuracy,Precision,Recall,F1-score,AUC
0,A_lr_0.001_ep10,0.753 ± 0.039,0.682 ± 0.088,0.761 ± 0.083,0.711 ± 0.016,0.835 ± 0.006
1,B_lr_0.0005_ep10,0.786 ± 0.018,0.776 ± 0.05,0.66 ± 0.073,0.709 ± 0.031,0.837 ± 0.014


In [ ]:

#CALIBRACIÓN

calibration_table = summary_df[[
    "config_name",
    "brier_mean", "brier_std",
    "logloss_mean", "logloss_std"
]].copy()

# Formato media ± std
calibration_table["Brier Score"] = (
    calibration_table["brier_mean"].round(3).astype(str)
    + " ± " +
    calibration_table["brier_std"].round(3).astype(str)
)

calibration_table["LogLoss"] = (
    calibration_table["logloss_mean"].round(3).astype(str)
    + " ± " +
    calibration_table["logloss_std"].round(3).astype(str)
)

# Seleccionar columnas finales
calibration_table = calibration_table[[
    "config_name",
    "Brier Score",
    "LogLoss"
]]

calibration_table

,config_name,Brier Score,LogLoss
0,A_lr_0.001_ep10,0.192 ± 0.028,0.723 ± 0.138
1,B_lr_0.0005_ep10,0.185 ± 0.015,1.016 ± 0.123


In [ ]:
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay

output_dir = "/content/drive/MyDrive/TFG/modelo_robusto_5folds"

results_df = pd.read_csv(f"{output_dir}/robust_5fold_gridsearch_results.csv")
summary_df = pd.read_csv(f"{output_dir}/robust_5fold_summary_mean_std.csv")
threshold_summary_df = pd.read_csv(f"{output_dir}/robust_5fold_threshold_summary_mean_std.csv")

selected_config = "A_lr_0.001_ep10"
selected_threshold = 0.4

In [ ]:
threshold_table = threshold_summary_df[
    threshold_summary_df["config_name"] == selected_config
].copy()

threshold_table = threshold_table[[
    "threshold",
    "accuracy_mean", "accuracy_std",
    "precision_mean", "precision_std",
    "recall_mean", "recall_std",
    "f1_mean", "f1_std"
]]

threshold_table["Threshold"] = threshold_table["threshold"]

for metric, name in [
    ("accuracy", "Accuracy"),
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("f1", "F1-score")
]:
    threshold_table[name] = (
        threshold_table[f"{metric}_mean"].round(3).astype(str)
        + " ± " +
        threshold_table[f"{metric}_std"].round(3).astype(str)
    )

threshold_table = threshold_table[[
    "Threshold", "Accuracy", "Precision", "Recall", "F1-score"
]]

threshold_table.to_csv(f"{output_dir}/tabla_thresholds.csv", index=False)
threshold_table

,Threshold,Accuracy,Precision,Recall,F1-score
0,0.3,0.712 ± 0.057,0.616 ± 0.082,0.815 ± 0.075,0.695 ± 0.027
1,0.4,0.735 ± 0.048,0.65 ± 0.086,0.788 ± 0.078,0.705 ± 0.023
2,0.5,0.753 ± 0.039,0.682 ± 0.088,0.761 ± 0.083,0.711 ± 0.016


In [ ]:
threshold_table = threshold_summary_df[
    threshold_summary_df["config_name"] == selected_config
].copy()

threshold_table = threshold_table[[
    "threshold",
    "accuracy_mean", "accuracy_std",
    "precision_mean", "precision_std",
    "recall_mean", "recall_std",
    "f1_mean", "f1_std"
]]

threshold_table["Threshold"] = threshold_table["threshold"]

for metric, name in [
    ("accuracy", "Accuracy"),
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("f1", "F1-score")
]:
    threshold_table[name] = (
        threshold_table[f"{metric}_mean"].round(3).astype(str)
        + " ± " +
        threshold_table[f"{metric}_std"].round(3).astype(str)
    )

threshold_table = threshold_table[[
    "Threshold", "Accuracy", "Precision", "Recall", "F1-score"
]]

threshold_table.to_csv(f"{output_dir}/tabla_thresholds.csv", index=False)
threshold_table

,Threshold,Accuracy,Precision,Recall,F1-score
0,0.3,0.712 ± 0.057,0.616 ± 0.082,0.815 ± 0.075,0.695 ± 0.027
1,0.4,0.735 ± 0.048,0.65 ± 0.086,0.788 ± 0.078,0.705 ± 0.023
2,0.5,0.753 ± 0.039,0.682 ± 0.088,0.761 ± 0.083,0.711 ± 0.016


In [ ]:


threshold_table = threshold_summary_df[[
    "config_name",
    "threshold",
    "accuracy_mean", "accuracy_std",
    "precision_mean", "precision_std",
    "recall_mean", "recall_std",
    "f1_mean", "f1_std"
]].copy()

# Renombrar configuraciones
threshold_table["Configuración"] = threshold_table["config_name"].replace({
    "A_lr_0.001_ep10": "LR = 0.001",
    "B_lr_0.0005_ep10": "LR = 0.0005"
})

# Formato media ± std
for metric, name in [
    ("accuracy", "Accuracy"),
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("f1", "F1-score")
]:
    threshold_table[name] = (
        threshold_table[f"{metric}_mean"].round(3).astype(str)
        + " ± " +
        threshold_table[f"{metric}_std"].round(3).astype(str)
    )

# Selección final columnas limpias
threshold_table = threshold_table[[
    "Configuración",
    "threshold",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score"
]]

# Renombrar threshold
threshold_table = threshold_table.rename(columns={
    "threshold": "Threshold"
})

threshold_table

,Configuración,Threshold,Accuracy,Precision,Recall,F1-score
0,LR = 0.001,0.3,0.712 ± 0.057,0.616 ± 0.082,0.815 ± 0.075,0.695 ± 0.027
1,LR = 0.001,0.4,0.735 ± 0.048,0.65 ± 0.086,0.788 ± 0.078,0.705 ± 0.023
2,LR = 0.001,0.5,0.753 ± 0.039,0.682 ± 0.088,0.761 ± 0.083,0.711 ± 0.016
3,LR = 0.0005,0.3,0.778 ± 0.019,0.74 ± 0.052,0.695 ± 0.074,0.713 ± 0.025
4,LR = 0.0005,0.4,0.783 ± 0.019,0.761 ± 0.05,0.676 ± 0.073,0.712 ± 0.029
5,LR = 0.0005,0.5,0.786 ± 0.018,0.776 ± 0.05,0.66 ± 0.073,0.709 ± 0.031
